# 00. 준비 확인 (CPU, 약 3분)

다른 계정에서 처음 돌릴 때 한 번 실행한다. 파일이 제자리에 있는지, 판정할 칸이 몇 개인지 본다. GPU 필요 없음.

**전체 순서** (각 노트북은 따로 실행, 단계 사이 데이터는 드라이브 파일로 넘김)

| 노트북 | 하는 일 | 런타임 | 시간(예상) |
|---|---|---|---|
| 00 | 준비 확인 | CPU | 3분 |
| 01 | Qwen으로 **Val** 칸 판정 + 검수 파일 | A100 | 15~30분 |
| (검수) | 01의 검수 파일을 Claude/사람이 확인 → 못 믿을 증상은 '삭제만'이나 '안 함'으로 | - | - |
| 02 | Qwen으로 **Train** 칸 판정 (끊기면 이어서) | A100 | 1~2시간 (01이 실제 시간을 알려 줌) |
| 03 | 라벨 수정 + 애매한 칸 삭제 + 증상별 분포표 | CPU | 2분 |
| 04 | KoELECTRA 최종 학습(8개 증상) + 점수표 | A100 | 정리 모델 40분 (+원본 모델 40분) |
| 05 | 시연: 통화 하나 넣고 증상 예측 | GPU 권장 | 2분 |

**폴더 구조**
```
증상인식_최종_프로젝트/
├── src/        기능 코드 (config, data, text, judge_prompt, judge_llm, cells, cleaning, model, trainer, metrics, valfix, evaluate, infer)
├── notebooks/  00~05
├── data_in/    labels_df.pkl (필수), val_claude_judge.xlsx (선택) ← 읽기만 함
├── judge/      Qwen 판정 결과 (01, 02)
├── data/       고친 라벨·삭제 마스크 (03)
├── results/    점수표, 요약, 검수 파일
└── models/     학습한 모델 (04)
```

## 0. 준비

In [ ]:
# 드라이브 연결 + src 폴더를 import할 수 있게 경로 추가
from google.colab import drive
drive.mount("/content/drive")

import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/증상인식_최종_프로젝트"   # 폴더를 다른 곳에 두었으면 여기만 바꾸기
os.environ["SYMPTOM_FINAL_ROOT"] = PROJECT_ROOT
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
# src/*.py를 고친 뒤에는 런타임 → 세션 다시 시작 후 이 셀부터 다시 실행

In [ ]:
import pandas as pd
from src import config
from src.data import load_labels
from src.cells import build_cells
print("프로젝트 폴더:", config.PROJECT_ROOT)

## 1. 입력 파일 확인

In [ ]:
for p, need in [(config.LABELS_PKL, "필수"), (config.VAL_CLAUDE_JUDGE, "선택")]:
    print(f"[{need}] {'있음' if os.path.exists(p) else '없음'}  {p}")
for d in ["src", "notebooks"]:
    print(d, sorted(os.listdir(f"{config.PROJECT_ROOT}/{d}")))

## 2. 데이터 읽기

In [ ]:
tr_df, va_df = load_labels()
print("증상(8개):", config.LABELS)

## 3. 판정할 칸 세기
라벨 1인 칸 + 라벨 0이지만 관련 단어가 있는 칸만 Qwen에게 묻는다.

In [ ]:
va_cells = build_cells(va_df)
tr_cells = build_cells(tr_df)
chars = lambda c: c["대화"].str.len().sum()
print(f"\nVal {len(va_cells):,}칸 / Train {len(tr_cells):,}칸 (Train이 Val의 {len(tr_cells) / len(va_cells):.1f}배)")
print(f"평균 녹취록 길이(글자): Val {chars(va_cells) / len(va_cells):.0f}, Train {chars(tr_cells) / len(tr_cells):.0f}")
print("→ 01에서 Val 걸린 시간 × 위 배수 ≈ 02(Train) 시간")